In [2]:
import json
import pandas as pd
from pathlib import Path


In [3]:
input_path = Path("../data/risk_dataset_v2.csv")

df = pd.read_csv(input_path)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (61, 7)


,license,license_family,project_type,distribution_model,usage,risk,reason
0,BSD-3-Clause,Permissive,commercial closed-source,distributed,library,Low,BSD license is permissive and mainly requires ...
1,BSD-2-Clause,Permissive,commercial SaaS,hosted,library,Low,BSD-2-Clause has minimal compliance obligations
2,MIT,Permissive,internal enterprise tool,internal-only,library,Low,Internal use with MIT license is generally low...
3,Apache-2.0,Permissive,open-source project,distributed,framework,Low,Apache-2.0 is permissive and suitable for open...
4,Apache-2.0,Permissive,mobile application,distributed,library,Low,Apache license allows commercial use with attr...


In [4]:
required_columns = [
    "license",
    "license_family",
    "project_type",
    "distribution_model",
    "usage",
    "risk",
    "reason"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

print("All required columns are present.")

All required columns are present.


In [5]:
def build_user_prompt(row):
    return f"""
You are an open-source license compliance assistant.

Classify the compliance risk for the following software dependency scenario.

License: {row["license"]}
License Family: {row["license_family"]}
Project Type: {row["project_type"]}
Distribution Model: {row["distribution_model"]}
Usage: {row["usage"]}

Return exactly:
Risk: Low/Medium/High
Reason: one short sentence
""".strip()


def build_assistant_response(row):
    return f"""Risk: {row["risk"]}
Reason: {row["reason"]}"""

In [6]:
records = []

for _, row in df.iterrows():
    sample = {
        "messages": [
            {
                "role": "user",
                "content": build_user_prompt(row)
            },
            {
                "role": "assistant",
                "content": build_assistant_response(row)
            }
        ]
    }

    records.append(sample)

print("Total fine-tuning samples:", len(records))
records[0]

Total fine-tuning samples: 61


{'messages': [{'role': 'user',
   'content': 'You are an open-source license compliance assistant.\n\nClassify the compliance risk for the following software dependency scenario.\n\nLicense: BSD-3-Clause\nLicense Family: Permissive\nProject Type: commercial closed-source\nDistribution Model: distributed\nUsage: library\n\nReturn exactly:\nRisk: Low/Medium/High\nReason: one short sentence'},
  {'role': 'assistant',
   'content': 'Risk: Low\nReason: BSD license is permissive and mainly requires retaining notices'}]}

In [7]:
output_path = Path("../data/qwen_finetune_dataset.jsonl")

with open(output_path, "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved fine-tuning dataset to: {output_path}")

Saved fine-tuning dataset to: ../data/qwen_finetune_dataset.jsonl


In [8]:
validation_records = []

with open(output_path, "r", encoding="utf-8") as f:
    for line in f:
        validation_records.append(json.loads(line))

print("Loaded records:", len(validation_records))
print(validation_records[0])

Loaded records: 61
{'messages': [{'role': 'user', 'content': 'You are an open-source license compliance assistant.\n\nClassify the compliance risk for the following software dependency scenario.\n\nLicense: BSD-3-Clause\nLicense Family: Permissive\nProject Type: commercial closed-source\nDistribution Model: distributed\nUsage: library\n\nReturn exactly:\nRisk: Low/Medium/High\nReason: one short sentence'}, {'role': 'assistant', 'content': 'Risk: Low\nReason: BSD license is permissive and mainly requires retaining notices'}]}


In [9]:
train_df = df.sample(frac=0.8, random_state=42)
val_df = df.drop(train_df.index)

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))

Train samples: 49
Validation samples: 12


In [10]:
def dataframe_to_jsonl(dataframe, path):
    records = []

    for _, row in dataframe.iterrows():
        sample = {
            "messages": [
                {
                    "role": "user",
                    "content": build_user_prompt(row)
                },
                {
                    "role": "assistant",
                    "content": build_assistant_response(row)
                }
            ]
        }

        records.append(sample)

    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"Saved {len(records)} samples to {path}")

In [11]:
dataframe_to_jsonl(train_df, "../data/qwen_finetune_train.jsonl")
dataframe_to_jsonl(val_df, "../data/qwen_finetune_val.jsonl")

Saved 49 samples to ../data/qwen_finetune_train.jsonl
Saved 12 samples to ../data/qwen_finetune_val.jsonl
